# Fundamentos Teóricos de la Regresión Logística

La regresión logística es un método estadístico utilizado para modelar la probabilidad de que un evento binario ocurra. Es decir, se utiliza para problemas de clasificación donde la variable dependiente es categórica (por ejemplo, sobrevivir o no sobrevivir).

### Ecuación de la Regresión Logística
La regresión logística utiliza la función sigmoide para modelar probabilidades:

$$ \hat{y} = \frac{1}{1 + e^{-z}} $$

Donde:
- $\hat{y}$: Probabilidad estimada de que el evento ocurra.
- $z$: Combinación lineal de las variables independientes ($z = \beta_0 + \beta_1x_1 + \beta_2x_2 + \dots + \beta_nx_n$).

### Supuestos de la Regresión Logística
1. La variable dependiente es binaria.
2. Las observaciones son independientes entre sí.
3. No hay multicolinealidad entre las variables independientes.
4. Relación lineal entre las variables independientes y el logit de la variable dependiente.

### Fundamento Matemático
La regresión logística maximiza la verosimilitud de los datos observados. La función de verosimilitud es:

$$ L(\beta) = \prod_{i=1}^n \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i} $$

Donde $y_i$ son los valores observados y $\hat{y}_i$ son las probabilidades predichas. El objetivo es encontrar los coeficientes $\beta$ que maximizan esta función.

### Fundamento Estadístico
La regresión logística asume que el logit (logaritmo de las probabilidades) sigue una relación lineal con las variables independientes:

$$ \text{logit}(\hat{y}) = \ln\left(\frac{\hat{y}}{1 - \hat{y}}\right) = \beta_0 + \beta_1x_1 + \beta_2x_2 + \dots + \beta_nx_n $$

Esto permite interpretar los coeficientes como el cambio en el logit por unidad de cambio en la variable independiente.

# ¿Cómo funciona la Regresión Logística?

La regresión logística es un modelo de clasificación que predice la probabilidad de que una observación pertenezca a una clase específica. Utiliza la función sigmoide para transformar una combinación lineal de las variables independientes en una probabilidad.

---

## Pasos de la Regresión Logística

- **1. Transformar las probabilidades en logits:**  
  La regresión logística modela el logit (logaritmo de las probabilidades):

$$
  \text{logit}(\hat{y}) = \ln\left(\frac{\hat{y}}{1 - \hat{y}}\right)
$$

- **2. Ajustar los coeficientes:**  
  Los coeficientes $\beta$ se ajustan para maximizar la función de verosimilitud, que mide qué tan bien el modelo predice los datos observados.

- **3. Predecir probabilidades:**  
  Una vez ajustado el modelo, se calculan las probabilidades de pertenecer a la clase positiva:

$$
  \hat{y} = \frac{1}{1 + e^{-z}}
$$

- **4. Clasificar observaciones:**  
  Se utiliza un umbral (por defecto 0.5) para clasificar las observaciones en una clase u otra.

---

## Ejemplo práctico

Supón que estás prediciendo si un pasajero del Titanic sobrevivió o no basado en su edad, clase y género. El modelo ajustado podría ser:

$$
\text{logit}(\hat{y}) = -1.5 + 0.02 \cdot \text{edad} - 0.8 \cdot \text{clase} + 2.3 \cdot \text{género}
$$

Esto significa que:
- Cada año adicional de edad reduce ligeramente la probabilidad de sobrevivir.
- Estar en una clase más alta aumenta la probabilidad de sobrevivir.
- Ser mujer aumenta significativamente la probabilidad de sobrevivir.

---

Este proceso permite interpretar los coeficientes y predecir probabilidades para nuevas observaciones.

In [1]:
# Conectar a la base de datos y realizar el query para obtener el dataset del Titanic
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/titanic.db"
r = requests.get(url)

with open("titanic.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("titanic.db")

query = """
SELECT
    O.survived,
    O.pclass,
    O.age,
    O.sibsp,
    O.parch,
    O.fare,
    O.adult_male,
    O.alone,
    S.sex,
    E.embarked
FROM
    Observation AS O
JOIN
    Sex AS S ON O.sex_id = S.sex_id
JOIN
    Embarked AS E ON O.embarked_id = E.embarked_id
"""
df = pd.read_sql_query(query, conn)
df.head()

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,sex,embarked
0,0,3,22.0,1,0,7.2500,1,0,male,S
1,1,1,38.0,1,0,71.2833,0,0,female,C
2,1,3,26.0,0,0,7.9250,0,1,female,S
3,1,1,35.0,1,0,53.1000,0,0,female,S
4,0,3,35.0,0,0,8.0500,1,1,male,S


In [2]:
# Dividir los datos en entrenamiento y prueba
from sklearn.model_selection import train_test_split

X = df.drop(columns=['survived'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(712, 9)
(179, 9)


In [3]:
X_train.isnull().sum()

pclass          0
age           140
sibsp           0
parch           0
fare            0
adult_male      0
alone           0
sex             0
embarked        2
dtype: int64

In [4]:
# Distribución de edad
import plotly.express as px

fig = px.histogram(
    data_frame=X_train,
    x='age',
    nbins=30,
    title='Distribución y densidad de la Edad',
    marginal="box"
)

print("mean: ", str(X_train.age.mean()))
print("median: ", str(X_train.age.median()))

fig.show()

mean:  29.498846153846156
median:  28.0


In [5]:
pct_faltantes = ( X_train.embarked.isnull().sum() / X_train.shape[0] ) * 100
print("Porcentaje de faltantes en embarked", str(pct_faltantes))

Porcentaje de faltantes en embarked 0.2808988764044944


In [6]:
# Mostrar conteo de pasajeros por puerto de embarque
print('Pasajeros embarcados agrupados por puerto')
print('C = Cherburgo')
print('Q = Queenstown')
print('S = Southampton')
print()
print(X_train.embarked.value_counts())

# Crear gráfico de barras con Plotly
fig = px.histogram(
    data_frame=X_train,
    x='embarked',
    color='embarked',
    title='Distribución de pasajeros por puerto de embarque'
)

fig.show()

Pasajeros embarcados agrupados por puerto
C = Cherburgo
Q = Queenstown
S = Southampton

embarked
S    525
C    125
Q     60
Name: count, dtype: int64


In [7]:
# Contar valores de supervivencia
conteo_supervivencia = y_train.value_counts().rename({0: 'No Sobrevivió', 1: 'Sobrevivió'})

# Crear DF para el gráfico
df_pie = pd.DataFrame(
    {
        'Estado': conteo_supervivencia.index,
        'Cantidad': conteo_supervivencia.values
    }
)
df_pie

# Crear gráfico de pastel
fig = px.pie(df_pie, names='Estado', values='Cantidad',
             title='Distribución de supervivencia en el conjunto de entrenamiento',
             color_discrete_sequence=['lightcoral', 'darkturquoise'],
             hole=0.3)

fig.update_traces(textinfo='percent+label')
fig.show()

In [8]:
train_data = X_train.copy()

train_data["age"] = train_data["age"].fillna(
    train_data["age"].median(skipna=True)
)

train_data["embarked"] = train_data["embarked"].fillna(
    train_data['embarked'].value_counts().idxmax()
)

In [9]:
train_data.isnull().sum()

pclass        0
age           0
sibsp         0
parch         0
fare          0
adult_male    0
alone         0
sex           0
embarked      0
dtype: int64

In [10]:
import numpy as np

train_data['TravelAlone'] = np.where(
    (train_data["sibsp"] + train_data["parch"]) > 0,
    0,
    1
)

train_data.drop('sibsp', axis=1, inplace=True)
train_data.drop('parch', axis=1, inplace=True)
display(train_data)

,pclass,age,fare,adult_male,alone,sex,embarked,TravelAlone
331,1,45.5,28.5000,1,1,male,S,1
733,2,23.0,13.0000,1,1,male,S,1
382,3,32.0,7.9250,1,1,male,S,1
704,3,26.0,7.8542,1,0,male,S,0
813,3,6.0,31.2750,0,0,female,S,0
...,...,...,...,...,...,...,...,...
106,3,21.0,7.6500,0,1,female,S,1
270,1,28.0,31.0000,1,1,male,S,1
860,3,41.0,14.1083,1,0,male,S,0
435,1,14.0,120.0000,0,0,female,S,0


In [11]:
# Vamos a crear variables dummies
from sklearn.preprocessing import OneHotEncoder

# Definir variables categóricas
categorical_cols = ["pclass", "embarked", "sex"]

# Inicializar OHE
encoder = OneHotEncoder(sparse_output=False, drop=None)

# Ajustar y transformar
encoded = encoder.fit_transform(train_data[categorical_cols])

# Crear tabla de dummies
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=train_data.index
)

encoded_df

,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S,sex_female,sex_male
331,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
733,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
382,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
704,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
813,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
270,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
860,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
435,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [12]:
# Concatenar con las columnas originales
training = pd.concat(
    [
        train_data.drop(columns=categorical_cols),
        encoded_df
    ],
    axis=1
)

# Eliminar una columna que estorba
training.drop('adult_male', axis=1, inplace=True)

final_train = training
final_train.head()

,age,fare,alone,TravelAlone,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S,sex_female,sex_male
331,45.5,28.5000,1,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
733,23.0,13.0000,1,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
382,32.0,7.9250,1,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
704,26.0,7.8542,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
813,6.0,31.2750,0,0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [13]:
X_test.isnull().sum()

pclass         0
age           37
sibsp          0
parch          0
fare           0
adult_male     0
alone          0
sex            0
embarked       0
dtype: int64

In [14]:
# Se utiliza la información de Train Data para evitar 'data leakage'

test_data = X_test.copy()
test_data["age"] = test_data["age"].fillna(X_train["age"].median(skipna=True))

test_data['TravelAlone'] = np.where(
    (test_data["sibsp"]+test_data["parch"])>0,
    0, 
    1
)

test_data.drop('sibsp', axis=1, inplace=True)
test_data.drop('parch', axis=1, inplace=True)

encoded_test = encoder.transform(test_data[categorical_cols])

# Crear DataFrame con nombres de columnas
encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=test_data.index
)

# Concatenar con las columnas numéricas originales
testing = pd.concat([test_data.drop(columns=categorical_cols), encoded_test_df], axis=1)

# Eliminar columna que no quieras
testing.drop('adult_male', axis=1, inplace=True)

final_test = testing
final_test.head()

,age,fare,alone,TravelAlone,pclass_1,pclass_2,pclass_3,embarked_C,embarked_Q,embarked_S,sex_female,sex_male
709,28.0,15.2458,0,0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
439,31.0,10.5000,1,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
840,20.0,7.9250,1,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
720,6.0,33.0000,0,0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
39,14.0,11.2417,0,0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [15]:
# Gráfico simple de la distribución de edad según supervivencia
import plotly.express as px

df = final_train.copy()
df['survived'] = y_train

fig = px.histogram(
    df, 
    x='age', 
    color='survived',
    nbins=30,
    barmode='overlay',
    labels={'survived': 'Supervivencia', 'age': 'Edad'},
    title='Distribución de edad según supervivencia'
)

fig.show()

In [16]:
final_train['IsMinor'] = np.where(
    final_train['age']<=16,
    1, 
    0
)

final_test['IsMinor'] = np.where(
    final_test['age']<=16, 
    1, 
    0
)

In [17]:
df = final_train.copy()
df['survived'] = y_train

fig = px.histogram(
    df,
    x='fare',
    color='survived',
    nbins=30,
    barmode='overlay',
    labels={'survived': 'Supervivencia', 'fare': 'Tarifa'},
    title='Distribución de tarifas según supervivencia',
    range_x=[0, 200]  # Limitar el rango para mejor visualización
)

fig.show()

In [18]:
def graficar_supervivencia_por_categoria(X_train, y_train, col_categorica):
    """
    Genera un gráfico de barras que muestra la tasa de supervivencia promedio
    según una variable categórica específica.

    Parámetros:
    - X_train: DataFrame con las variables explicativas
    - y_train: Serie o DataFrame con la variable 'survived'
    - col_categorica: string con el nombre de la columna categórica de interés
    """

    # Asegurar que los índices coincidan
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    # Combinar datos
    df = X_train.copy()
    df['survived'] = y_train

    # Agrupar por la variable categórica y calcular promedio de supervivencia
    promedio = df.groupby(col_categorica, as_index=False)['survived'].mean()

    # Títulos dinámicos
    titulo = f'Tasa de supervivencia por categoría: {col_categorica}'
    etiqueta_x = col_categorica.capitalize()
    etiqueta_y = 'Tasa de supervivencia'

    # Crear gráfico
    fig = px.bar(promedio, x=col_categorica, y='survived',
                 title=titulo,
                 labels={col_categorica: etiqueta_x, 'survived': etiqueta_y},
                 color_discrete_sequence=['darkturquoise'],
                 width=800, height=400)

    fig.show()

In [19]:
graficar_supervivencia_por_categoria(
    X_train,
    y_train,
    'embarked'
)

## Regresión Logística

In [20]:
from sklearn.linear_model import LogisticRegression

# Seleccionar columnas relevantes
columnas = [
    "age",
    "fare",
    "TravelAlone",
    "pclass_1",
    "pclass_2",
    "embarked_C",
    "embarked_S",
    "sex_male",
    "IsMinor"
]

X = final_train[columnas]
y = y_train

modelo = LogisticRegression(max_iter=500)
modelo

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",500
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in

In [21]:
from sklearn.feature_selection import RFECV
import plotly.graph_objects as go

# Aplicar RFECV con regresión logística
rfecv = RFECV(
    estimator=modelo, 
    step=1, 
    cv=10, 
    scoring='accuracy'
)
rfecv.fit(X, y)

# Mostrar resultados
print(f'Número óptimo de variables seleccionadas: {rfecv.n_features_}')
print(f'Variables seleccionadas: {list(X.columns[rfecv.support_])}')

# Obtener puntajes de validación cruzada desde cv_results_
scores = rfecv.cv_results_['mean_test_score']
num_features = list(range(1, len(scores) + 1))

# Crear gráfico interactivo
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=num_features,
    y=scores,
    mode='lines+markers',
    line=dict(color='darkturquoise', width=3),
    marker=dict(size=6),
    name='Puntaje de validación cruzada'
))

fig.update_layout(
    title='Optimización de selección de variables con RFECV',
    xaxis_title='Número de variables seleccionadas',
    yaxis_title='Puntaje de validación cruzada (accuracy)',
    width=900,
    height=500,
    template='plotly_white'
)

fig.show()

Número óptimo de variables seleccionadas: 8
Variables seleccionadas: ['age', 'TravelAlone', 'pclass_1', 'pclass_2', 'embarked_C', 'embarked_S', 'sex_male', 'IsMinor']


In [22]:
# Crear lista de variables seleccionadas
# Seleccionar variables
selected_features = ['age', 'TravelAlone', 'pclass_1', 'pclass_2', 'embarked_C', 
                     'embarked_S', 'sex_male', 'IsMinor']
X = final_train[selected_features]

# Entrenar modelo de regresión logística
X_train = final_train[selected_features]
X_test = final_test[selected_features]

modelo = LogisticRegression(max_iter=500)
modelo.fit(X_train, y_train)

# Predicciones
y_pred = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]

In [23]:
from sklearn.metrics import classification_report, confusion_matrix

lr_cr = classification_report(y_test, y_pred)
print(lr_cr)

              precision    recall  f1-score   support

           0       0.82      0.84      0.83       105
           1       0.76      0.74      0.75        74

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [24]:
# Matriz de confusión con Plotly
cm_lr = confusion_matrix(y_test, y_pred)

cm_lr_df = pd.DataFrame(
    cm_lr,
    index=['Real: No sobrevivió', 'Real: Sobrevivió'],
    columns=['Pred: No sobrevivió', 'Pred: Sobrevivió']
)

fig = px.imshow(
    cm_lr_df,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de confusión - Regresión Logística'
)
fig.update_layout(width=700, height=450)
fig.show()

# Modelo Alternativo 2: Decision Tree Classifier

In [25]:
from sklearn.tree import DecisionTreeClassifier

# Ajustar arbol para encontrar IMPORTANCIAS
dt = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

dt.fit(X_train, y_train)

dt

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

In [26]:
# crear dataframe de importancia
importances = dt.feature_importances_
importances

array([0.28757514, 0.028265  , 0.07315251, 0.10387685, 0.00922279,
       0.03796415, 0.44963084, 0.0103127 ])

In [27]:
importancias = pd.DataFrame(
    {
        'Variable': X_train.columns,
        'Importancia': importances
    }
).sort_values('Importancia', ascending=False)

importancias

,Variable,Importancia
6,sex_male,0.449631
0,age,0.287575
3,pclass_2,0.103877
2,pclass_1,0.073153
5,embarked_S,0.037964
1,TravelAlone,0.028265
7,IsMinor,0.010313
4,embarked_C,0.009223


In [28]:
fig = px.bar(
    data_frame=importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='Importances - Decision Tree'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [29]:
selected_vars = importancias.sort_values(
    'Importancia',
    ascending=False
).assign(
    Cum=lambda df: df['Importancia'].cumsum()
).loc[
    lambda df: df['Cum'] <=0.95, 'Variable'
].tolist()

selected_vars


['sex_male', 'age', 'pclass_2', 'pclass_1']

In [30]:
dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=2
)

dt.fit(
    X_train[selected_vars], 
    y_train
)

dt

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samp

In [31]:
# Predecir
y_pred_dt = dt.predict(X_test[selected_vars])

# Classification report
dt_cr = classification_report(y_test, y_pred_dt)
print(dt_cr)

              precision    recall  f1-score   support

           0       0.80      0.87      0.83       105
           1       0.78      0.69      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



In [32]:
# Matriz de confusión

cm_dt = confusion_matrix(y_test, y_pred_dt)

cm_dt_df = pd.DataFrame(
    data=cm_dt,
    index=['Real: No Sobrevivió', 'Real: Sobrevivió'],
    columns=['Pred: No sobrevivió', 'Pred: Sobrevivió']
)

fig = px.imshow(
    cm_dt_df,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de confusión Decision Tree'
)
fig.show()

Modelo Alternativo - KNN

In [33]:
from sklearn.neighbors import KNeighborsClassifier

# Entrenar KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predicciones
y_pred_knn = knn.predict(X_test)

# Classification report
knn_cr = classification_report(y_test, y_pred_knn)
print(knn_cr)

              precision    recall  f1-score   support

           0       0.74      0.86      0.80       105
           1       0.74      0.58      0.65        74

    accuracy                           0.74       179
   macro avg       0.74      0.72      0.72       179
weighted avg       0.74      0.74      0.74       179

